In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
from torch.utils.data import TensorDataset, DataLoader, random_split
result_df = pd.read_csv("drive/MyDrive/202509CURRENTcell_embeddings.csv") #change it to your classification probability cell csv
result_df.head() #shape should like this:

,x,y,region,prob_class0,prob_class1,prob_class2,prob_class3,prob_class4,prob_class5,prob_class6,...,prob_class15,prob_class16,prob_class17,prob_class18,prob_class19,prob_class20,prob_class21,prob_class22,prob_class23,prob_class24
0,434,382,B004_Ascending,0.000168,0.000005,0.158509,7.306978e-08,0.000004,0.000012,4.156397e-06,...,8.825942e-08,0.840157,0.000613,0.000118,0.000011,1.186485e-07,0.000048,0.000039,0.000009,0.000021
1,565,465,B004_Ascending,0.027036,0.036189,0.000456,4.775565e-01,0.018246,0.008158,4.710367e-07,...,8.924435e-04,0.004983,0.133771,0.001888,0.003422,6.794071e-04,0.000515,0.050863,0.096157,0.004591
2,661,355,B004_Ascending,0.000258,0.000472,0.028910,9.215923e-06,0.000096,0.000827,2.182230e-04,...,3.623293e-05,0.477269,0.002370,0.002763,0.000231,5.750678e-06,0.000992,0.005804,0.000193,0.000057
3,826,267,B004_Ascending,0.015109,0.005193,0.008532,8.633848e-05,0.001027,0.004175,5.199927e-05,...,5.246982e-05,0.473917,0.001660,0.001053,0.000220,7.540340e-06,0.003047,0.065482,0.027111,0.002067
4,739,439,B004_Ascending,0.027876,0.059822,0.000107,2.242883e-01,0.015348,0.004501,2.172979e-07,...,4.064996e-04,0.009830,0.184847,0.000790,0.001863,5.420372e-04,0.000319,0.201525,0.198495,0.000835


In [ ]:
# autoencoder model
class Autoencoder(nn.Module):
    def __init__(self, in_dim=25, bottleneck_dim=3, hidden_dim=512):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, int(hidden_dim/4)),
            nn.ReLU(),
            nn.LayerNorm(int(hidden_dim/4)),
            nn.Dropout(0.1),
            nn.Linear(int(hidden_dim/4), int(hidden_dim/2)),
            nn.ReLU(),
            nn.LayerNorm(int(hidden_dim/2)),
            nn.Dropout(0.1),
            nn.Linear(int(hidden_dim/2), int(hidden_dim)),
            nn.ReLU(),
            nn.LayerNorm(int(hidden_dim)),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, int(hidden_dim/2)),
            nn.ReLU(),
            nn.LayerNorm(int(hidden_dim/2)),
            nn.Dropout(0.1),
            nn.Linear(int(hidden_dim/2), int(hidden_dim/4)),
            nn.ReLU(),
            nn.LayerNorm(int(hidden_dim/4)),
            nn.Dropout(0.1),
            nn.Linear(int(hidden_dim/4), bottleneck_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(bottleneck_dim, int(hidden_dim/4)),
            nn.ReLU(),
            nn.LayerNorm(int(hidden_dim/4)),
            nn.Dropout(0.1),
            nn.Linear(int(hidden_dim/4), int(hidden_dim/2)),
            nn.ReLU(),
            nn.LayerNorm(int(hidden_dim/2)),
            nn.Dropout(0.1),
            nn.Linear(int(hidden_dim/2), hidden_dim),
            nn.ReLU(),
            nn.LayerNorm(int(hidden_dim)),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, int(hidden_dim/2)),
            nn.ReLU(),
            nn.LayerNorm(int(hidden_dim/2)),
            nn.Dropout(0.1),
            nn.Linear(int(hidden_dim/2), int(hidden_dim/4)),
            nn.ReLU(),
            nn.LayerNorm(int(hidden_dim/4)),
            nn.Dropout(0.1),
            nn.Linear(int(hidden_dim/4), in_dim)
        )

    def forward(self, x):
        z = self.encoder(x)
        return z, self.decoder(z)



In [ ]:
# Dataset of all emebddings
emb_matrix = result_df[[col for col in result_df.columns if col.startswith("prob_")]].values # shape = [N, hidden_dim], N = total nodes from all graphs
emb_matrix_tensor = torch.tensor(emb_matrix, dtype=torch.float32)
emb_matrix_tensor

In [ ]:
# loss function

def bio_contrastive_loss(z, orig_probs, margin=50.0, alpha=0.1):
    """
    Biologically weighted contrastive loss.

    Goal:
      - Keep samples of the same cell type close together (intra-class cohesion)
      - Push different cell types apart (inter-class separation)
      - Allow biologically similar cell types (based on probability similarity)
        to be closer in the latent space

    Args:
        z (torch.Tensor): Latent embeddings of shape [B, D]
        orig_probs (torch.Tensor): Cell type probability distributions [B, C]
        margin (float): Margin distance for inter-class separation
        alpha (float): Strength of biological similarity weighting
    """
    B = z.size(0)
    device = z.device

    # --- Step 1: Compute pairwise Euclidean distances in latent space ---
    dists = torch.cdist(z, z, p=2)  # [B, B]

    # --- Step 2: Identify same-type and different-type pairs ---
    labels = orig_probs.argmax(dim=1)               # Hard label per sample
    same = (labels.unsqueeze(1) == labels.unsqueeze(0))  # [B, B]
    diff = ~same

    # --- Step 3: Compute biological similarity between cells ---
    # Use cosine similarity between cell-type probability distributions
    # to reflect biological closeness between different types
    prob_sim = F.cosine_similarity(
        orig_probs.unsqueeze(1),  # [B, 1, C]
        orig_probs.unsqueeze(0),  # [1, B, C]
        dim=-1
    )  # [B, B]
    prob_sim = torch.clamp(prob_sim, 0, 1)  # Ensure range [0, 1]

    # --- Step 4: Intra-class loss (same type) ---
    # Encourage embeddings of the same cell type to be close together
    intra_loss = dists[same].sum() / (same.sum() + 1e-8)

    # --- Step 5: Inter-class loss (different types) ---
    # Encourage different types to be far apart,
    # but scale the penalty by biological similarity
    # (more similar types get a smaller penalty)
    inter_weight = (1 - alpha * prob_sim[diff])
    inter_loss = (inter_weight * torch.clamp(margin - dists[diff], min=0)).sum() / (diff.sum() + 1e-8)

    # --- Step 6: Combine total loss ---
    total_loss = 0.1 * intra_loss + inter_loss

    return total_loss


In [ ]:
def loss_function(orig_probs, pred_logits, z, margin=50.0, beta=0.1):
    pred_log_probs = F.log_softmax(pred_logits, dim=1)
    recon_loss = F.kl_div(pred_log_probs, orig_probs, reduction='batchmean')
    cluster_loss = bio_contrastive_loss(z, orig_probs, margin)
    total_loss = recon_loss + beta * cluster_loss
    return total_loss, recon_loss.item(), cluster_loss.item()


In [ ]:
# train autoencoder

dataset1 = TensorDataset(emb_matrix_tensor)

val_ratio = 0.1
val_size = int(len(dataset1) * val_ratio)
train_size = len(dataset1) - val_size
train_set1, val_set1 = random_split(dataset1, [train_size, val_size])

batch_size = 4096
train_loader1 = DataLoader(train_set1, batch_size=batch_size, shuffle=True, num_workers=4)
val_loader1 = DataLoader(val_set1, batch_size=batch_size, shuffle=False, num_workers=4)
device = torch.device('cuda')
model = Autoencoder().to("cuda")
optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)
#model.load_state_dict(torch.load("drive/MyDrive/newae1.pth"))

num_epochs = 20
alpha = 0.1
from tqdm import tqdm

for epoch in range(num_epochs):

    # ------- train -------
    model.train()
    total_loss = 0
    total_recon = 0
    total_div = 0
    t_correct = 0
    t_total = 0
    for batch in tqdm(train_loader1, desc=f"Train Epoch {epoch+1}"):
        x = batch[0].to(device)
        optimizer.zero_grad()
        z, out = model(x)
        loss, recon_loss, diversity_loss = loss_function(x, out, z, beta=alpha)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        total_recon += recon_loss
        total_div += diversity_loss
        out_label = out.argmax(dim=1)
        x_label = x.argmax(dim=1)
        t_correct += (out_label == x_label).sum().item()
        t_total += x.size(0)
    val_acc = t_correct / t_total
    total_loss /= len(train_loader1)
    total_recon /= len(train_loader1)
    total_div /= len(train_loader1)
    print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {total_loss:.4f} | Recon: {total_recon:.4f} | Diversity: {total_div:.4f} | Train Acc: {val_acc:.4f}")


    # ------- validate -------
    model.eval()
    val_loss = 0
    val_recon = 0
    val_div = 0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for batch in tqdm(val_loader1, desc=f"Val Epoch {epoch+1}"):
            x = batch[0].to(device)
            z, out = model(x)
            loss, recon_loss, diversity_loss = loss_function(x, out, z, beta=alpha)
            val_loss += loss.item()
            val_recon += recon_loss
            val_div += diversity_loss
            x_label = x.argmax(dim=1)
            out_label = out.argmax(dim=1)
            val_correct += (out_label == x_label).sum().item()
            val_total += x.size(0)
    val_acc = val_correct / val_total
    val_loss /= len(val_loader1)
    val_recon /= len(val_loader1)
    val_div /= len(val_loader1)
    print(f"Epoch {epoch+1}/{num_epochs} | Val Loss: {val_loss:.4f} | Recon: {val_recon:.4f} | Diversity: {val_div:.4f} | Val Acc: {val_acc:.4f}")
    torch.save(model.state_dict(), "your_path")

In [ ]:
# convert to rgb

model.eval()
with torch.no_grad():
    x_in = torch.tensor(emb_matrix, dtype=torch.float32).to("cuda")
    model = model.to("cuda")
    z_3d = model.encoder(x_in).cpu().numpy()  # shape [N, 3]

# z_3d can have arbitrary range; let's min-max scale each dimension
min_vals = z_3d.min(axis=0)
max_vals = z_3d.max(axis=0)
print(min_vals, max_vals) # IMPORTANT!!!!!!! you need this scale for interpret

# Avoid zero-division if max == min
range_vals = (max_vals - min_vals)
#range_vals[range_vals == 0] = 1e-9

scaled_3d = (z_3d - min_vals) / range_vals  # [0, 1]
rgb_3d = (scaled_3d * 255).astype(np.uint8) # [0..255]

# Add columns back to df
result_df["R"] = rgb_3d[:, 0]
result_df["G"] = rgb_3d[:, 1]
result_df["B"] = rgb_3d[:, 2]

# Save to csv
result_df.to_csv("drive/MyDrive/node_embeddings_3d.csv", index=False)

[-69.761505 -75.65188  -77.16103 ] [88.969406 65.244896 67.13518 ]


In [ ]:
# save embedded images

import cv2
import os

save_dir = "drive/MyDrive/region_images" #change to your path
os.makedirs(save_dir, exist_ok=True)

all_regions = result_df["region"].unique()
cnt = 0
for reg in all_regions:
    subset = result_df[result_df["region"] == reg]
    xs = subset["x"].values
    ys = subset["y"].values
    Rs = subset["R"].values
    Gs = subset["G"].values
    Bs = subset["B"].values

    # Create a blank image (adjust size if needed)
    # Example: 1024 x 1024, 3 channels
    img = np.ones((1024, 1024, 3), dtype=np.uint8) * 255

    for i in range(len(subset)):
        x_int = int(round(xs[i]))
        y_int = int(round(ys[i]))
        # Safety check
        if 0 <= x_int < 1024 and 0 <= y_int < 1024:
            img[y_int, x_int, 0] = Bs[i]  # OpenCV uses BGR order by default
            img[y_int, x_int, 1] = Gs[i]
            img[y_int, x_int, 2] = Rs[i]

    # Save the image
    save_path = os.path.join(save_dir, f"region_{cnt}.png")
    cv2.imwrite(save_path, img)
    cnt += 1

    print(f"Saved image for region {cnt} -> {save_path}")
